# Strain Gages and Murphy's Law
### *Modern Strain Gage Technology* — Companion Notebook

**Companion notebook to Chapter 9 — Strain Gages and Murphy's Law.**

*Anything that can go wrong will go wrong* — so a dependable measurement system is one that makes going wrong hard by design, not merely unlikely by luck.

*Murphy's Law* — "anything that can go wrong, will go wrong" — is often quoted as cynical humor. In engineering it is a design principle. A system that permits incorrect assembly will eventually be assembled incorrectly. A calibration procedure that can be skipped will eventually be skipped. A data format that can be misinterpreted will eventually be misinterpreted. Murphy's Law is not about pessimism; it is about recognizing that human reliability has limits and designing measurement systems that are robust against predictable failures.

For strain gage installations, the failure mode inventory is well-established: poor surface preparation leading to debonds, lead wire shorts to the structure, incorrect bridge wiring, wrong gage factor entered in software, shunt calibration omitted, anti-aliasing filter not configured, zero not recorded before loading. Each failure mode has an independent probability of occurring per installation. When eight independent modes each carry a 2–10% probability, the probability that everything is perfectly correct drops below 75% before the test even begins.

The solution is not more careful engineers — it is system design. Checklist-driven procedures, automated configuration verification, wiring configurations that fail visibly rather than silently, and software that requires the user to confirm critical settings before recording begins are all engineering solutions to the human reliability problem. This notebook quantifies the compounding effect of independent failure modes and makes the case for process design as the primary quality tool.

**This notebook demonstrates:**
1. Cumulative failure probability — as independent failure modes are added one at a time, the probability of a flawless measurement degrades steadily from ~100% toward an unacceptable floor

In [ ]:
# !pip install numpy matplotlib   # Colab: uncomment to install dependencies
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 12})

# Output directory for the figure the book includes — do not rename.
OUT_DIR = Path("../src/media/ch09")
OUT_DIR.mkdir(parents=True, exist_ok=True)

---
## 1 — Cumulative Failure Probability: Murphy's Law Made Quantitative

The probability that a strain gage installation is completely free of any failure mode is the product of the individual "did not happen" probabilities across every independent mode:

$$P(\text{all OK}) = \prod_{i=1}^{n}\left(1 - p_i\right)$$

In words: multiply together the chance that each failure mode *didn't* occur, where $p_i$ is the probability that mode $i$ happens on a given installation. Because every factor is less than one, the product shrinks with each mode added — even when no single $p_i$ is large. Its complement, $1 - P(\text{all OK})$, is the probability that *at least one* thing went wrong. For the eight modes below, with individual probabilities ranging from 0.5% to 10%, $P(\text{all OK})$ works out to about 73% — meaning roughly one in four installations carries at least one active problem before the first data point is recorded.

The step chart accumulates failure modes one at a time in the order listed, showing how quickly the "all OK" probability degrades. Each step down represents the addition of one more independent failure mode to the inventory. The filled area above the curve represents the cumulative probability that at least one failure has occurred — the complement of "all OK." Note that no single failure mode has a large probability; the compound effect comes from their independence.

These numbers are illustrative; real probabilities depend heavily on training, procedure quality, and installation complexity. The key insight is independent of the specific values: when many independent failure modes exist, the compound probability of a perfect outcome is substantially lower than any individual success probability. This is the mathematical argument for systematic process design — not as overhead, but as the primary mechanism for keeping compound failure probability at an acceptable level.

In [ ]:
# Failure modes and their probability of occurring per installation.
# Illustrative values — replace with rates from your own process.
FAILURE_MODES = {
    "Gage debonded (poor surface prep)":     0.02,
    "Lead wire short to structure":          0.01,
    "Bridge wired incorrectly":              0.03,
    "Excitation polarity reversed":          0.005,
    "Wrong gage factor entered in software": 0.04,
    "Shunt cal not performed":               0.10,
    "Anti-alias filter not configured":      0.08,
    "Zero not recorded before loading":      0.05,
}

FIG_PATH = OUT_DIR / "fig09_nb1_murphys_law_probability.png"
LABEL_MAX_CHARS = 30   # truncate long mode names on the x-axis


def probability_all_ok(failure_probs):
    """Probability that *every* independent failure mode is avoided.

    Returns the product of (1 - p) over all modes — the chance a single
    installation is free of every listed defect.
    """
    return float(np.prod(1.0 - np.asarray(failure_probs)))


def cumulative_all_ok(failure_probs):
    """P(all OK) as failure modes are added one at a time.

    Element k is the product of (1 - p) over the first k modes, so the
    curve shows how the 'perfect installation' probability compounds
    downward as the failure-mode inventory grows.
    """
    return np.cumprod(1.0 - np.asarray(failure_probs))


names = list(FAILURE_MODES.keys())
probs = np.array(list(FAILURE_MODES.values()))

# Probability that ALL modes are fine, and its complement.
p_all_ok = probability_all_ok(probs)
p_at_least_one_failure = 1.0 - p_all_ok

print(f"P(all modes OK)           = {p_all_ok:.3f}  ({p_all_ok*100:.1f}%)")
print(f"P(at least one failure)   = {p_at_least_one_failure:.3f}  ({p_at_least_one_failure*100:.1f}%)")
print()

# Cumulative: add modes one at a time to show the compounding effect.
p_ok_cumulative = cumulative_all_ok(probs)

fig, ax = plt.subplots(figsize=(10, 4))
ax.step(range(1, len(names)+1), p_ok_cumulative * 100, 'steelblue', lw=2, where='post')
ax.fill_between(range(1, len(names)+1), p_ok_cumulative*100, 100, alpha=0.1, color='tomato',
                label='Probability of ≥1 failure')
ax.set_xticks(range(1, len(names)+1))
ax.set_xticklabels([n.split('(')[0].strip()[:LABEL_MAX_CHARS] for n in names],
                   rotation=40, ha='right', fontsize=8)
ax.set_ylabel('P(all OK) %'); ax.set_xlabel('Failure modes added cumulatively')
ax.set_title("Murphy's Law: Cumulative Probability of Perfect Measurement")
ax.set_ylim(0, 105); ax.legend(); ax.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout(); plt.savefig(FIG_PATH, bbox_inches='tight', dpi=150)
plt.show()
print("Murphy's insight: systems that PERMIT incorrect assembly will eventually be assembled incorrectly.")
print("The solution is design — not more careful people.")

---
## Where to take this next

This model is a foundation, not a finished tool. A few concrete extensions:

- **Plug in your own numbers.** Replace the illustrative rates in `FAILURE_MODES` with values from your own installation records or lab QA logs, then watch how the `P(all OK)` floor moves. Which single mode dominates the loss, and is it the one you would have guessed?
- **Quantify the payoff of a checklist.** Give each mode a *mitigation factor* — say, a checklist step that cuts its probability by 5–10× — and re-plot `P(all OK)` with and without it. This turns the chapter's qualitative argument for process design into a number: the recovered probability of a clean measurement.
- **Go from a point estimate to a distribution.** Treat each $p_i$ as uncertain (draw it from a Beta distribution) and run a Monte Carlo simulation to get a *distribution* of `P(all OK)` rather than a single value. Seed the generator with `np.random.default_rng(0)` so the run is reproducible — a first step toward the formal uncertainty budgets developed in Part 8.